# Reading Serial Data from Arduino

This notebook shows the simplest way to read data from an Arduino (or any serial device) using Python.

We do everything step by step:

1. Install the [`pyserial`](https://pyserial.readthedocs.io/en/latest/index.html) library  
2. Open the serial port  
3. Read one line  
4. (Optional) Read lines in a loop  
5. Close the serial port



## Install Library

```pip install pyserial```


## Open the Serial Port

Change `"COM3"` to the correct port name.

If this cell runs without an error, the port is open and you can start reading.

In [12]:
import serial

# Change this to your own port:
PORT = "COM3"
BAUD = 9600

ser = serial.Serial(PORT, BAUD, timeout=0.1)
print("Serial port opened:", ser)


Serial port opened: Serial<id=0x22062c9bdf0, open=True>(port='COM3', baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=0.1, xonxoff=False, rtscts=False, dsrdtr=False)


## Read One Line

This reads only one line from the Arduino.

If the Arduino is sending text using `Serial.println()`, it will show up here.

You can run this cell again and again to read more lines.


In [15]:
raw = ser.readline()  # read raw bytes
text = raw.decode(errors="ignore").strip()  # convert to text
print("Received:", text)


Received: 0.41


## Read Continuously (Simple Loop)

This loop prints new lines from the Arduino as they arrive.

`time.sleep(0.1)` slows down the loop so it is easy to read.

Stop the loop with **Kernel → Interrupt**.


In [16]:
import time

while True:
    raw = ser.readline()
    text = raw.decode(errors="ignore").strip()
    print(text)
    time.sleep(0.1)


0.50
0.44
0.39
0.35
0.37
0.30
0.32
0.35
0.36
0.31
0.30
0.39
0.38
0.36
0.39
0.48
0.45
0.42
0.39
0.46
0.42
0.45
0.51
0.46
0.48


KeyboardInterrupt: 

## Close the Serial Port

Always close the port when you are finished.

If you forget to close it, sometimes Windows/Mac/Linux refuses to open the port again until you disconnect the Arduino.


In [17]:
ser.close()
print("Port closed")


Port closed


---

# Read Serial Data and Save to Pandas CSV

This example reads data from an Arduino using a serial connection.
It then:

1. Stores each received value in a pandas DataFrame  
2. Adds a timestamp  
3. Saves the DataFrame into a CSV file  
4. Allows repeated reading and growing of the table  

We keep everything in small steps.

## Open the Serial Port

Change the port name to match your system.

If this cell runs without errors, the port is open.


In [18]:
import serial

PORT = "COM3"   # <-- change this
BAUD = 9600

ser = serial.Serial(PORT, BAUD, timeout=0.1)
print("Serial port opened:", ser)


Serial port opened: Serial<id=0x22062db4610, open=True>(port='COM3', baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=0.1, xonxoff=False, rtscts=False, dsrdtr=False)


## Create an Empty Pandas DataFrame

We create a DataFrame that will store:
- the timestamp
- the Arduino value

Each time we read new data, we add a new row to this DataFrame.


In [19]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(columns=["timestamp", "value"])

df


,timestamp,value


## Read One Line and Add It to the DataFrame

This cell:
1. Reads one line from the serial port
2. Converts it to text
3. Creates a timestamp
4. Appends the data to the DataFrame
5. Shows the updated table

Run this cell many times to collect more rows.


In [20]:
raw = ser.readline()
text = raw.decode(errors="ignore").strip()

if text != "":
    new_row = {
        "timestamp": datetime.now().isoformat(),
        "value": text
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

df.tail()   # show last rows


,timestamp,value
0,2025-11-30T17:26:41.601465,-0.23


## Read Continuously and Add to the DataFrame

This loop keeps reading lines and adds them to the DataFrame.
Stop it with **Kernel → Interrupt**.

Use this if you want the notebook to collect data automatically.


In [21]:
import time

print("Reading... Press Kernel → Interrupt to stop.")

while True:
    raw = ser.readline()
    text = raw.decode(errors="ignore").strip()

    if text != "":
        new_row = {
            "timestamp": datetime.now().isoformat(),
            "value": text
        }
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

    time.sleep(0.1)  # slow it down a bit


Reading... Press Kernel → Interrupt to stop.


KeyboardInterrupt: 

In [22]:
df

,timestamp,value
0,2025-11-30T17:26:41.601465,-0.23
1,2025-11-30T17:27:01.726730,-0.15
2,2025-11-30T17:27:01.828180,-0.17
3,2025-11-30T17:27:01.930543,-0.27
4,2025-11-30T17:27:02.035091,-0.18
...,...,...
65,2025-11-30T17:27:08.287077,0.39
66,2025-11-30T17:27:08.390982,0.42
67,2025-11-30T17:27:08.492797,0.47
68,2025-11-30T17:27:08.594914,0.54


## Save the DataFrame to a CSV

This exports all collected data to a CSV file.

You can open the CSV in:
- Excel
- Google Sheets
- Any data analysis tool


In [10]:
df.to_csv("arduino_data.csv", index=False)
print("Saved to arduino_data.csv")


Saved to arduino_data.csv


## Close the Serial Port


In [23]:
ser.close()
print("Port closed")


Port closed


In [6]:
running = False      # tell the thread to stop
sock.close()         # unblock recvfrom and close the socket
print("UDP listener stopped and socket closed.")


UDP listener stopped and socket closed.
